In [1]:
import zstandard as zstd

def decompress_zstd_file(input_file_path, output_file_path):
    # Create the decompressor context
    dctx = zstd.ZstdDecompressor()
    
    # Open the compressed file for reading in binary mode
    with open(input_file_path, 'rb') as compressed_file:
        # Open the target file for writing in binary mode
        with open(output_file_path, 'wb') as output_file:
            # Efficiently stream the decompression
            dctx.copy_stream(compressed_file, output_file)

# Usage example:
decompress_zstd_file('../data/raw/databento/ONDS/xnas-itch-20260325.mbo.ONDS.csv.zst', '../data/raw/databento/ONDS/20260325_ONDS.csv')

In [2]:
import pandas as pd
df = pd.read_csv('../data/raw/databento/ONDS/20260325_ONDS.csv')
df.head()

,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,price,size,channel_id,order_id,flags,ts_in_delta,sequence,symbol
0,1774422303606439871,1774422303606276714,160,2,11989,R,N,9223372036854775807,0,0,0,8,0,0,ONDS
1,1774425600235047455,1774425600234883829,160,2,11989,A,A,14440000000,2,0,36931,128,163626,314246,ONDS
2,1774425600655742848,1774425600655578320,160,2,11989,A,A,16000000000,78,0,56475,128,164528,319684,ONDS
3,1774425600756843952,1774425600756680822,160,2,11989,A,B,10500000000,166,0,60647,128,163130,321335,ONDS
4,1774425600761214280,1774425600761051451,160,2,11989,A,A,15200000000,35,0,60747,128,162829,321424,ONDS


In [11]:
df[df["action"]=="T"].head()

,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,price,size,channel_id,order_id,flags,ts_in_delta,sequence,symbol
63,1774425606029218190,1774425606029054506,160,2,11989,T,N,11010000000,437,0,0,128,163684,369034,ONDS
64,1774425606029224182,1774425606029060764,160,2,11989,T,N,11010000000,454,0,0,128,163418,369035,ONDS
71,1774425606029368353,1774425606029204885,160,2,11989,T,B,11020000000,194,0,0,0,163468,369049,ONDS
74,1774425606029440306,1774425606029274774,160,2,11989,T,N,11010000000,24,0,0,128,165532,369055,ONDS
75,1774425606029444232,1774425606029280997,160,2,11989,T,B,11020000000,24,0,0,0,163235,369057,ONDS


In [39]:
valid_ids = df.groupby("order_id")["action"].apply(lambda x: set(x) == {'C', 'F'})
df[df["order_id"].isin(valid_ids[valid_ids].index)]

,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,price,size,channel_id,order_id,flags,ts_in_delta,sequence,symbol


In [10]:
df[df["order_id"]==69919]

,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,price,size,channel_id,order_id,flags,ts_in_delta,sequence,symbol
9,1774425600978747208,1774425600978584226,160,2,11989,A,A,11030000000,693,0,69919,128,162982,324255,ONDS
86,1774425606029538548,1774425606029373592,160,2,11989,F,A,11030000000,157,0,69919,0,164956,369075,ONDS
87,1774425606029538548,1774425606029373592,160,2,11989,C,A,11030000000,157,0,69919,128,164956,369075,ONDS
91,1774425606029580319,1774425606029416882,160,2,11989,F,A,11030000000,536,0,69919,0,163437,369080,ONDS
92,1774425606029580319,1774425606029416882,160,2,11989,C,A,11030000000,536,0,69919,128,163437,369080,ONDS


In [9]:
df[df["sequence"]==369049]

,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,price,size,channel_id,order_id,flags,ts_in_delta,sequence,symbol
71,1774425606029368353,1774425606029204885,160,2,11989,T,B,11020000000,194,0,0,0,163468,369049,ONDS
72,1774425606029368353,1774425606029204885,160,2,11989,F,A,11020000000,194,0,153767,0,163468,369049,ONDS
73,1774425606029368353,1774425606029204885,160,2,11989,C,A,11020000000,194,0,153767,128,163468,369049,ONDS


In [3]:
import sys
sys.path.insert(0, "/home/leo/lob_research")

import pandas as pd
from src.ingestion.coinbase import process

events = process("/home/leo/lob_research/data/raw/coinbase/20260323_BTC-USDT.csv")

dups = events[events.duplicated(subset=["order_id", "session_id", "event_seq"], keep=False)]
print(dups.sort_values(["order_id", "session_id", "event_seq"]).head(20).to_string())

2026-04-04 17:56:26 [info     ] parsed filename                date=20260323 source=coinbase symbol=BTC-USDT


2026-04-04 17:56:43 [info     ] loaded raw csv                 path=/home/leo/lob_research/data/raw/coinbase/20260323_BTC-USDT.csv rows=13213166
2026-04-04 17:57:24 [info     ] midnight correction applied    date_today=2026-03-23 date_yesterday=2026-03-22 rows_corrected=3333 ts_col=time_exchange
